# PlanTo3D — train the segmenter on CubiCasa5K

Trains a U-Net with a ResNet34 encoder to label floor plan pixels as
background, wall, door, window, and **what each room is for** — bedroom,
kitchen, bath, storage, circulation or outdoor.

**Set the runtime to GPU first:** Runtime → Change runtime type → T4 GPU.

## What changed since the last checkpoint

Two things, and both are aimed at the same problem: reading a plan the
model has never seen.

**It predicts the room type now.** An earlier model predicted one
undifferentiated "room" and left the question of what a room was *for* to
OCR. That works on a drawing with room names printed on it. Most drawings
have none: over sixty CubiCasa plans, OCR read a room name on three. Since
floor finishes, planting, railings and wet areas all hang off knowing what
a room is for, a text-only route leaves the great majority of plans bare.

**The training data is augmented.** There was no augmentation at all, which
is the largest thing that was missing. A model shown 5,000 plans learns
those 5,000 plans; what makes it read a sixth thousand is having been shown
each of them drawn differently — turned, mirrored, at another size, dimmer,
softer, and put through JPEG. Every one of those corresponds to something
that actually arrives at the pipeline and is currently read badly.

## Order

Sections 1–5 set up and then *verify against real data* before section 7
spends GPU hours. The annotation parser is unit-tested against synthetic
SVG only, so a real sample is the first genuine check that the room types
come through correctly. A wrong mapping trains a model that looks broken
for reasons that have nothing to do with training.


## 1. Confirm the GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("NO GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")

## 2. Install dependencies and fetch the code

In [ ]:
!pip install -q segmentation-models-pytorch kagglehub opencv-python-headless

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/priyanshsoni096-blip/PlanTo3D.git"
repo = Path("/content/PlanTo3D")

# Fetched and reset rather than pulled. A pull merges, and merging fails
# outright if the remote history has been rewritten -- which it has, to
# strip some personal floor plans out of it. Reset takes whatever is on
# the remote and asks no questions, which is what you want here: there is
# nothing in this clone worth keeping.
if repo.exists():
    !cd {repo} && git fetch --quiet origin && git reset --hard --quiet origin/main && git clean -qfd
else:
    !git clone --quiet {REPO_URL} {repo}

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

# Anything imported before this cell re-ran is the old code still in
# memory. Colab keeps modules loaded across cell runs, so a fresh pull
# changes the files on disk and nothing else.
for name in [m for m in sys.modules if m.startswith(("planto3d", "training"))]:
    del sys.modules[name]

from planto3d.classes import CLASS_NAMES

!cd {repo} && git log --oneline -1

print(f"
{len(CLASS_NAMES)} classes: {list(CLASS_NAMES.values())}")

## 3. Download CubiCasa5K

About 6 GB, pulled straight into the Colab VM rather than uploaded from your
machine. `kagglehub` will ask you to authenticate on first use.

In [ ]:
import kagglehub

download_root = Path(kagglehub.dataset_download("qmarva/cubicasa5k"))
print(f"downloaded to {download_root}")

# The archive nests a second cubicasa5k folder; the split files mark the root
# the sample paths in train.txt are relative to.
candidates = [p.parent for p in download_root.rglob("train.txt")]
if not candidates:
    raise SystemExit(f"no train.txt found under {download_root}")

DATA_ROOT = candidates[0]
print(f"data root: {DATA_ROOT}")
print(f"contents:  {sorted(p.name for p in DATA_ROOT.iterdir())[:10]}")

In [ ]:
from planto3d.cubicasa import sample_paths

splits = {
    name: sample_paths(DATA_ROOT, DATA_ROOT / f"{name}.txt")
    for name in ("train", "val", "test")
}
for name, pairs in splits.items():
    print(f"{name:6} {len(pairs):5d} samples")

## 4. Verify the annotation mapping on real data

**Do not skip this.** Everything downstream assumes CubiCasa's categories
land on the right classes. Check that walls trace walls, that doors and
windows appear at openings, and above all that **the room types are
plausible** — bedrooms where beds are drawn, kitchens at the counters, baths
at the fixtures.

A type landing in the wrong class is the expensive failure here: it trains
cleanly, scores well, and produces tiled bedrooms in the finished model.
Fix `planto3d/cubicasa.py` before training rather than after.


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

from planto3d.classes import CLASS_NAMES, NUM_CLASSES
from planto3d.cubicasa import class_distribution, svg_to_mask

# One colour per class. Room types are given distinct hues so a mis-mapped
# type shows up as the wrong colour in the wrong place rather than hiding
# inside a single "room" blue.
PALETTE = np.array(
    [
        [255, 255, 255],  # background
        [40, 40, 40],     # wall
        [150, 200, 255],  # room, of no particular type
        [255, 80, 80],    # door
        [80, 220, 120],   # window
        [190, 160, 235],  # bedroom
        [255, 190, 90],   # kitchen
        [90, 210, 220],   # bath
        [170, 170, 150],  # storage
        [235, 225, 130],  # circulation
        [120, 190, 120],  # outdoor
    ],
    dtype=np.uint8,
)
assert len(PALETTE) == NUM_CLASSES, "a class has no colour"

LEGEND = "  ".join(
    f"{CLASS_NAMES[i]}" for i in range(NUM_CLASSES) if i not in (0,)
)


def show(pairs, count=3):
    fig, axes = plt.subplots(count, 3, figsize=(15, 5 * count))
    axes = np.atleast_2d(axes)

    for row, (image_path, svg_path) in enumerate(pairs[:count]):
        image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
        mask = svg_to_mask(svg_path, image.shape[:2])
        coloured = PALETTE[mask]

        axes[row][0].imshow(image)
        axes[row][0].set_title(image_path.parent.name)
        axes[row][1].imshow(coloured)
        axes[row][1].set_title("annotation")
        axes[row][2].imshow(cv2.addWeighted(image, 0.55, coloured, 0.45, 0))
        axes[row][2].set_title("overlay")
        for ax in axes[row]:
            ax.axis("off")

        shares = class_distribution(mask)
        found = " ".join(
            f"{CLASS_NAMES[i]}={shares[i]:.1%}"
            for i in range(NUM_CLASSES)
            if shares[i] > 0.001
        )
        print(f"{image_path.parent.name}: {found}")

    plt.tight_layout()
    plt.show()


print(f"legend: {LEGEND}
")

# Architectural sheets resemble the Soni Residence drawings most closely.
architectural = [p for p in splits["train"] if "architectural" in str(p[0])]
print(f"{len(architectural)} architectural samples
")
show(architectural or splits["train"])


In [ ]:
# Class balance across a sample of the training set, and the loss weights
# derived from it. Doors and windows are a fraction of a percent each, and
# that imbalance is why the loss weights cross-entropy by class as well as
# pairing it with Dice.
#
# Compare these shares against training.train.CLASS_FREQUENCY. If they have
# drifted far, re-measure with scripts/class_balance.py and update the table
# rather than leaving the weights aimed at the wrong distribution.
import random

from training.train import CLASS_FREQUENCY, class_weights

random.seed(0)
totals = np.zeros(NUM_CLASSES)
checked = 0

for image_path, svg_path in random.sample(splits["train"], min(60, len(splits["train"]))):
    image = cv2.imread(str(image_path))
    if image is None:
        continue
    shares = class_distribution(svg_to_mask(svg_path, image.shape[:2]))
    totals += np.array([shares[i] for i in range(NUM_CLASSES)])
    checked += 1

measured = totals / max(checked, 1)
weights = class_weights()

print(f"over {checked} samples:
")
print(f"{'class':14}{'measured':>10}{'expected':>10}{'weight':>9}")
for index in range(NUM_CLASSES):
    print(
        f"{CLASS_NAMES[index]:14}{measured[index]:9.2%}"
        f"{CLASS_FREQUENCY.get(index, 0):9.2%}{weights[index]:9.2f}"
    )

if measured[1] < 0.005:
    print("
WARNING: almost no wall pixels. The mapping is probably wrong -- check section 4.")

missing = [CLASS_NAMES[i] for i in range(5, NUM_CLASSES) if measured[i] < 0.001]
if missing:
    print(f"
WARNING: no pixels for {', '.join(missing)}. Room types are not")
    print("coming through -- check the SPACE_MAP in planto3d/cubicasa.py.")


## 5. Look at the augmentation

**Worth thirty seconds.** These transforms are the difference between a
model that reads CubiCasa and one that reads a floor plan, and a broken one
is invisible in the loss curve — it just trains to a worse number and
nothing says why.

Check that the mask turns *with* the image every time. If a drawing is
rotated and its annotation is not, the model is being taught that walls
are wherever it likes.


In [ ]:
import numpy as np

from training.augment import augment
from training.dataset import CubiCasaDataset

preview = CubiCasaDataset(DATA_ROOT, DATA_ROOT / "train.txt", size=384)
raw_image, raw_mask = None, None

# Pull one sample straight from disk, before any transform.
image_path, svg_path = preview.samples[0]
raw = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
raw = cv2.resize(raw, (384, 384), interpolation=cv2.INTER_AREA)
mask = cv2.resize(
    svg_to_mask(svg_path, cv2.imread(str(image_path)).shape[:2]).astype(np.uint8),
    (384, 384),
    interpolation=cv2.INTER_NEAREST,
)

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for column in range(5):
    rng = np.random.default_rng(column)
    shown, shown_mask = (raw.copy(), mask.copy()) if column == 0 else augment(
        raw.copy(), mask.copy(), rng
    )
    axes[0][column].imshow(shown)
    axes[0][column].set_title("original" if column == 0 else f"draw {column}")
    axes[1][column].imshow(PALETTE[shown_mask])
    for row in (0, 1):
        axes[row][column].axis("off")

plt.tight_layout()
plt.show()

print("The mask must turn with the image. If it does not, stop here.")

## 6. Smoke run

Two epochs over a handful of samples. Proves the loop runs on this data
before committing to the full job — cheaper to fail here than an hour in.

In [ ]:
import logging

from training.train import train

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)

train(
    DATA_ROOT,
    "/content/smoke.pt",
    epochs=2,
    batch_size=4,
    size=256,
    limit=24,
    num_workers=2,
)

## 7. Full training run

**Train at 768 this time.** That is the reason to run this again.

A window is drawn about 4 pixels wide and arrives at the network 1.5
pixels wide at a 512 input. It is the model's weakest class by a wide
margin -- IoU 0.096, finding 37% of the windows there are, against 0.52
to 0.77 for everything else. Its loss weight is already the highest at
3.72, and no amount of weighting recovers detail that resampling removed.

| class | drawn | at 512 | at 768 |
| --- | --- | --- | --- |
| wall | 22px | 8.4px | 12.5px |
| door | 16px | 6.5px | 9.8px |
| **window** | **4px** | **1.5px** | **2.2px** |

The cost is roughly double: about 12 minutes an epoch on a T4, so 24
epochs is nearer 5 hours than 2.5. The batch size drops with it, which
the cell handles.

If 5 hours is too long, 16 epochs is a reasonable trade -- the last run
had flattened well before the end, gaining 0.004 Dice over its final
four epochs.

The checkpoint is written to Drive after every epoch that improves, so a
disconnect costs the epoch in progress and nothing else.

**Watch the per-class IoU, not the Dice.** Window IoU is the number this
run exists to move. If it has not risen well clear of 0.096, the extra
hours bought nothing and that is worth knowing.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

CHECKPOINT = Path("/content/drive/MyDrive/planto3d/unet_cubicasa.pt")
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
print(f"checkpoint will be written to {CHECKPOINT}")

In [ ]:
#@title Size and length, then run { run: "auto" }

# 768 is the reason to run this again. A window is 1.5 pixels wide at 512
# and 2.2 at 768 -- still thin, but half again as much of it, and windows
# are far and away the model's weakest class at an IoU of 0.096 against
# 0.52 to 0.77 for everything else.
#
# It costs roughly double: about 12 minutes an epoch on a T4 rather than
# 6, so 24 epochs is nearer 5 hours than 2.5. Drop to 16 epochs if that is
# too long -- the last run had flattened by 20 of 24, gaining 0.004 Dice
# over its final four.
input_size = 768  #@param [512, 640, 768] {type:"raw"}
epochs = 24  #@param {type:"slider", min:8, max:32, step:2}

from training.dataset import SUGGESTED_BATCH
from training.train import train

batch_size = SUGGESTED_BATCH.get(input_size, 4)
print(f"{input_size}px, batch {batch_size}, {epochs} epochs")

train(
    DATA_ROOT,
    CHECKPOINT,
    epochs=epochs,
    batch_size=batch_size,
    size=input_size,
    learning_rate=3e-4,
    augment=True,
)

## 8. Score the held-out test split

In [ ]:
from torch.utils.data import DataLoader

from training.dataset import CubiCasaDataset
from training.train import build_model, evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
state = torch.load(CHECKPOINT, map_location=device, weights_only=False)

# Built to the checkpoint's own class count rather than the current one, so
# a model trained before the room types existed can still be scored here.
model = build_model(state.get("num_classes", NUM_CLASSES)).to(device)
model.load_state_dict(state["model_state"])

test_loader = DataLoader(
    CubiCasaDataset(DATA_ROOT, DATA_ROOT / "test.txt", size=state["size"]),
    batch_size=8,
    num_workers=2,
)

metrics = evaluate(model, test_loader, device)
print(f"test Dice {metrics['dice']:.4f}   test IoU {metrics['iou']:.4f}
")
for name, value in metrics["per_class_iou"].items():
    print(f"  {name:13} IoU {value if value is None else round(value, 4)}")

# The room types are the point of this model, so check them specifically.
# A type at zero means it was never predicted -- more training, or a look
# at whether that type is present in the data at all.
room_types = [
    (name, value)
    for name, value in metrics["per_class_iou"].items()
    if name in {"bedroom", "kitchen", "bath", "storage", "circulation", "outdoor"}
]
if room_types:
    weakest = min(room_types, key=lambda pair: pair[1] or 0.0)
    print(f"
weakest room type: {weakest[0]} at IoU {weakest[1]}")
else:
    print("
This checkpoint predicts no room types -- it predates them.")


## 9. Try it on the real floor plan

The point of the whole exercise: does the trained model beat the classical
baseline on a drawing style it never saw in training? Upload a page image
exported by `scripts/crop_pages.py`.

In [ ]:
from google.colab import files

from planto3d.classical import classical_mask
from training.dataset import IMAGENET_MEAN, IMAGENET_STD

uploaded = files.upload()
page = cv2.cvtColor(cv2.imread(next(iter(uploaded))), cv2.COLOR_BGR2RGB)

size = state["size"]
resized = cv2.resize(page, (size, size), interpolation=cv2.INTER_AREA)
normalized = (resized.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
batch = torch.from_numpy(normalized).permute(2, 0, 1).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    predicted = model(batch).argmax(dim=1)[0].cpu().numpy()

# Back to the page's own resolution so it can be compared with the baseline.
predicted = cv2.resize(
    predicted.astype(np.uint8), (page.shape[1], page.shape[0]), interpolation=cv2.INTER_NEAREST
)
baseline = classical_mask(cv2.cvtColor(page, cv2.COLOR_RGB2BGR))

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for ax, content, title in zip(
    axes,
    [page, PALETTE[baseline], PALETTE[predicted]],
    ["floor plan", "classical baseline", "trained U-Net"],
):
    ax.imshow(content)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

# What the model made of the rooms. On a plan with no printed names this is
# the only thing standing between the finished model and bare grey floors.
print("room types found:")
for index in range(5, NUM_CLASSES):
    share = float((predicted == index).mean())
    if share > 0.001:
        print(f"  {CLASS_NAMES[index]:13} {share:6.2%} of the page")


## 10. Bring the checkpoint home

About 98 MB. Save it under `models/` in the project, then pass
`--checkpoint` to the pipeline to use the model in place of the baseline.

In [ ]:
files.download(str(CHECKPOINT))